In [0]:
# HIGH_GARDEN_CATALOG_PARAMETER
dbutils.widgets.text("catalog", "high_garden")
catalog = dbutils.widgets.get("catalog").strip() or "high_garden"
print(f"Using Unity Catalog: {catalog}")


In [0]:
import numpy as np
import pandas as pd
import mlflow

from pyspark.sql import functions as F

In [0]:
SOURCE_TABLE = (
    f"{catalog}.silver.coffee_consumption"
)

OUTPUT_TABLE = (
    f"{catalog}.gold.consumption_forecast"
)

MODEL_NAME = (
    f"{catalog}.ml.coffee_forecaster"
)

MODEL_ALIAS = "Champion"

MODEL_URI = (
    f"models:/{MODEL_NAME}@{MODEL_ALIAS}"
)

In [0]:
mlflow.set_registry_uri(
    "databricks-uc"
)

In [0]:
champion_model = (
    mlflow.pyfunc.load_model(
        MODEL_URI
    )
)

print(
    "Loaded model:",
    MODEL_URI
)

In [0]:
silver_df = spark.table(
    SOURCE_TABLE
)

print(
    "Rows:",
    silver_df.count()
)

In [0]:
latest_year = (
    silver_df
    .agg(
        F.max(
            "start_year"
        ).alias("latest_year")
    )
    .first()["latest_year"]
)

forecast_year = latest_year + 1

print(
    "Latest observed year:",
    latest_year
)

print(
    "Forecast year:",
    forecast_year
)

In [0]:
latest_observations = (
    silver_df
    .filter(
        F.col("start_year")
        == latest_year
    )
    .select(
        "country",
        "coffee_type",
        F.col(
            "domestic_consumption"
        ).alias(
            "lag_1"
        )
    )
)

In [0]:
assert (
    latest_observations.count()
    == 55
), "Expected 55 markets"

assert (
    latest_observations
    .filter(
        F.col("lag_1").isNull()
    )
    .count()
    == 0
), "Null lag_1 values detected"

assert (
    latest_observations
    .filter(
        F.col("lag_1") < 0
    )
    .count()
    == 0
), "Negative lag_1 values detected"

print(
    "Forecast input validation passed."
)

In [0]:
forecast_input_df = (
    latest_observations
    .toPandas()
)

In [0]:
display(
    forecast_input_df.head()
)

In [0]:
model_input = (
    forecast_input_df[
        ["lag_1"]
    ]
)

In [0]:
forecast_input_df[
    "prediction"
] = champion_model.predict(
    model_input
)

In [0]:
display(
    forecast_input_df.head(10)
)

In [0]:
forecast_input_df[
    "prediction"
] = np.clip(
    forecast_input_df[
        "prediction"
    ],
    a_min=0,
    a_max=None,
)

In [0]:
forecast_input_df[
    "forecast_start_year"
] = forecast_year

forecast_input_df[
    "forecast_end_year"
] = forecast_year + 1

forecast_input_df[
    "crop_year"
] = (
    str(forecast_year)
    +
    "/"
    +
    str(
        forecast_year + 1
    )[-2:]
)

forecast_input_df[
    "model_name"
] = MODEL_NAME

forecast_input_df[
    "model_alias"
] = MODEL_ALIAS

In [0]:
forecast_input_df = (
    forecast_input_df
    .rename(
        columns={
            "lag_1":
            "last_observed_consumption"
        }
    )
)

In [0]:
forecast_output_df = (
    forecast_input_df[
        [
            "country",
            "coffee_type",
            "crop_year",
            "forecast_start_year",
            "forecast_end_year",
            "last_observed_consumption",
            "prediction",
            "model_name",
            "model_alias",
        ]
    ]
    .copy()
)

In [0]:
display(
    forecast_output_df
)

In [0]:
assert (
    len(
        forecast_output_df
    )
    == 55
), "Expected 55 forecasts"

assert (
    forecast_output_df[
        "prediction"
    ].isna().sum()
    == 0
), "Null forecasts detected"

assert (
    forecast_output_df[
        "prediction"
    ].lt(0).sum()
    == 0
), "Negative forecasts detected"

print(
    "Forecast validation passed."
)

In [0]:
forecast_sdf = (
    spark.createDataFrame(
        forecast_output_df
    )
)

In [0]:
forecast_sdf = (
    forecast_sdf
    .withColumn(
        "generated_at",
        F.current_timestamp()
    )
)

In [0]:
(
    forecast_sdf.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        OUTPUT_TABLE
    )
)

In [0]:
display(spark.sql(f"""
SELECT *
FROM {catalog}.gold.consumption_forecast
ORDER BY prediction DESC;
"""))


In [0]:
display(spark.sql(f"""
SELECT
    COUNT(*) AS forecasts,
    MIN(prediction) AS min_forecast,
    MAX(prediction) AS max_forecast,
    SUM(prediction) AS total_forecast
FROM {catalog}.gold.consumption_forecast;
"""))


In [0]:
latest_actual_total = (
    silver_df
    .filter(
        F.col("start_year")
        == latest_year
    )
    .agg(
        F.sum(
            "domestic_consumption"
        ).alias("total")
    )
    .first()["total"]
)

forecast_total = (
    forecast_sdf
    .agg(
        F.sum(
            "prediction"
        ).alias("total")
    )
    .first()["total"]
)

print(
    "Latest actual:",
    latest_actual_total
)

print(
    "Next-year forecast:",
    forecast_total
)

In [0]:
backtest_df = (
    spark.table(
        f"{catalog}.gold.backtest_predictions"
    )
    .filter(
        F.col("model")
        == "naive"
    )
    .toPandas()
)

In [0]:
absolute_errors = np.abs(
    backtest_df["target"]
    -
    backtest_df["prediction"]
)

In [0]:
q90_error = np.quantile(
    absolute_errors,
    0.90
)

print(
    "90% empirical error:",
    q90_error
)

In [0]:
forecast_output_df[
    "lower_bound_90"
] = np.clip(
    forecast_output_df[
        "prediction"
    ]
    -
    q90_error,
    a_min=0,
    a_max=None,
)

forecast_output_df[
    "upper_bound_90"
] = (
    forecast_output_df[
        "prediction"
    ]
    +
    q90_error
)